<a href="https://colab.research.google.com/github/Dineeesh2906/Deep-learning---24BAD021/blob/main/dl5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ============================================================
# TASK A: DATASET PREPARATION
# Tiny Shakespeare Dataset from Kaggle
# Student Roll No: 24BAD021
# ============================================================

# Install required libraries
!pip install -q -U pandas tensorflow scikit-learn

# ------------------------------------------------------------
# IMPORT LIBRARIES
# ------------------------------------------------------------

import os
import zipfile
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from google.colab import files
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# STUDENT DETAILS
# ------------------------------------------------------------

ROLL_NO = "24BAD021"

print("=" * 60)
print("RNN TEXT GENERATION - TASK A")
print("Student Roll No:", ROLL_NO)
print("=" * 60)

# ------------------------------------------------------------
# SET SEED
# ------------------------------------------------------------

np.random.seed(42)
tf.random.set_seed(42)

# ------------------------------------------------------------
# STEP 1: UPLOAD KAGGLE ZIP
# ------------------------------------------------------------

print("\nPlease upload your Kaggle ZIP file")

uploaded = files.upload()

# Find ZIP file
zip_filename = None

for filename in uploaded.keys():
    if filename.lower().endswith(".zip"):
        zip_filename = filename
        break

if zip_filename is None:
    raise FileNotFoundError(
        "No ZIP file found. Please upload a .zip file."
    )

print("\nUploaded:", zip_filename)

# ------------------------------------------------------------
# STEP 2: EXTRACT ZIP
# ------------------------------------------------------------

extract_dir = "/content/tiny_shakespeare"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("\n✅ ZIP file extracted successfully!")

# ------------------------------------------------------------
# STEP 3: FIND CSV FILES
# ------------------------------------------------------------

csv_files = []

for root, dirs, filenames in os.walk(extract_dir):

    for filename in filenames:

        if filename.lower().endswith(".csv"):

            csv_files.append(
                os.path.join(root, filename)
            )

print("\nCSV files found:")

for file in csv_files:
    print("✔", file)

# ------------------------------------------------------------
# CHECK CSV FILES
# ------------------------------------------------------------

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV files were found inside the ZIP file."
    )

# ------------------------------------------------------------
# STEP 4: LOAD TRAIN / VALIDATION / TEST DATA
# ------------------------------------------------------------

train_file = None
validation_file = None
test_file = None

for file in csv_files:

    filename = os.path.basename(file).lower()

    if "train" in filename:
        train_file = file

    elif "validation" in filename or "valid" in filename:
        validation_file = file

    elif "test" in filename:
        test_file = file

# ------------------------------------------------------------
# DISPLAY FILE LOCATIONS
# ------------------------------------------------------------

print("\nDataset files:")

print("Train      :", train_file)
print("Validation :", validation_file)
print("Test       :", test_file)

# ------------------------------------------------------------
# STEP 5: READ CSV FILES
# ------------------------------------------------------------

train_df = pd.read_csv(train_file)

print("\n" + "=" * 60)
print("TRAINING DATASET")
print("=" * 60)

print("\nShape:", train_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nFirst 5 rows:")
print(train_df.head())

# ------------------------------------------------------------
# VALIDATION DATA
# ------------------------------------------------------------

if validation_file is not None:

    validation_df = pd.read_csv(
        validation_file
    )

    print("\n" + "=" * 60)
    print("VALIDATION DATASET")
    print("=" * 60)

    print("\nShape:", validation_df.shape)

    print("\nColumns:")
    print(validation_df.columns.tolist())

# ------------------------------------------------------------
# TEST DATA
# ------------------------------------------------------------

if test_file is not None:

    test_df = pd.read_csv(
        test_file
    )

    print("\n" + "=" * 60)
    print("TEST DATASET")
    print("=" * 60)

    print("\nShape:", test_df.shape)

    print("\nColumns:")
    print(test_df.columns.tolist())

# ------------------------------------------------------------
# STEP 6: FIND TEXT COLUMN
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SEARCHING FOR TEXT COLUMN")
print("=" * 60)

# Common names for text columns
possible_text_columns = [
    "text",
    "Text",
    "TEXT",
    "content",
    "Content",
    "sentence",
    "Sentence"
]

text_column = None

for column in train_df.columns:

    if column in possible_text_columns:

        text_column = column
        break

# If common name is not found,
# automatically use the first object/string column
if text_column is None:

    object_columns = train_df.select_dtypes(
        include=["object"]
    ).columns

    if len(object_columns) > 0:

        text_column = object_columns[0]

# Check whether text column was found
if text_column is None:

    raise ValueError(
        "Could not find a text column in train.csv."
    )

print("\n✅ Text column found:")
print(text_column)

# ------------------------------------------------------------
# STEP 7: COMBINE TRAINING TEXT
# ------------------------------------------------------------

text = " ".join(
    train_df[text_column]
    .dropna()
    .astype(str)
    .tolist()
)

print("\n" + "=" * 60)
print("TEXT INFORMATION")
print("=" * 60)

print("\nTotal characters:", len(text))

print("\nFirst 1000 characters:")
print("-" * 60)

print(text[:1000])

print("-" * 60)

# ------------------------------------------------------------
# STEP 8: LOWERCASE
# ------------------------------------------------------------

text = text.lower()

print("\n✅ Text converted to lowercase.")

# ------------------------------------------------------------
# STEP 9: TOKENIZATION
# ------------------------------------------------------------

tokenizer = Tokenizer(
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'
)

tokenizer.fit_on_texts([text])

word_sequences = tokenizer.texts_to_sequences(
    [text]
)[0]

# ------------------------------------------------------------
# STEP 10: VOCABULARY
# ------------------------------------------------------------

vocab_size = len(
    tokenizer.word_index
) + 1

print("\n" + "=" * 60)
print("VOCABULARY")
print("=" * 60)

print("\nVocabulary Size:", vocab_size)

print(
    "Total Tokens:",
    len(word_sequences)
)

# Display sample vocabulary
print("\nFirst 20 vocabulary words:")

for word, index in list(
    tokenizer.word_index.items()
)[:20]:

    print(
        f"{word:15} -> {index}"
    )

# ------------------------------------------------------------
# STEP 11: CREATE SEQUENCES
# ------------------------------------------------------------

sequence_length = 20

print("\n" + "=" * 60)
print("SEQUENCE GENERATION")
print("=" * 60)

print(
    "\nSequence length:",
    sequence_length
)

input_sequences = []

for i in range(
    sequence_length,
    len(word_sequences)
):

    sequence = word_sequences[
        i - sequence_length : i + 1
    ]

    input_sequences.append(
        sequence
    )

input_sequences = np.array(
    input_sequences
)

print(
    "\nInput sequence shape:",
    input_sequences.shape
)

# ------------------------------------------------------------
# STEP 12: INPUT AND TARGET
# ------------------------------------------------------------

X = input_sequences[:, :-1]

y = input_sequences[:, -1]

print("\nX shape:", X.shape)

print("y shape:", y.shape)

# ------------------------------------------------------------
# STEP 13: TRAIN / VALIDATION SPLIT
# ------------------------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\n" + "=" * 60)
print("TRAIN / VALIDATION SPLIT")
print("=" * 60)

print(
    "\nTraining samples:",
    len(X_train)
)

print(
    "Validation samples:",
    len(X_val)
)

print(
    "\nTraining input shape:",
    X_train.shape
)

print(
    "Validation input shape:",
    X_val.shape
)

# ------------------------------------------------------------
# STEP 14: REVERSE VOCABULARY
# ------------------------------------------------------------

index_word = {
    index: word
    for word, index
    in tokenizer.word_index.items()
}

# ------------------------------------------------------------
# STEP 15: DISPLAY SAMPLE
# ------------------------------------------------------------

sample_index = 0

sample_words = [
    index_word.get(
        word_id,
        "<UNK>"
    )
    for word_id in X[sample_index]
]

target_word = index_word.get(
    y[sample_index],
    "<UNK>"
)

print("\n" + "=" * 60)
print("SAMPLE NEXT-WORD PREDICTION")
print("=" * 60)

print("\nInput sequence:")

print(
    " ".join(sample_words)
)

print("\nTarget / Next Word:")

print(target_word)

# ------------------------------------------------------------
# TASK A COMPLETE
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✅ TASK A COMPLETED SUCCESSFULLY")
print("=" * 60)

RNN TEXT GENERATION - TASK A
Student Roll No: 24BAD021

Please upload your Kaggle ZIP file


Saving archive (20).zip to archive (20) (3).zip

Uploaded: archive (20) (3).zip

✅ ZIP file extracted successfully!

CSV files found:
✔ /content/tiny_shakespeare/train.csv
✔ /content/tiny_shakespeare/validation.csv
✔ /content/tiny_shakespeare/test.csv

Dataset files:
Train      : /content/tiny_shakespeare/train.csv
Validation : /content/tiny_shakespeare/validation.csv
Test       : /content/tiny_shakespeare/test.csv

TRAINING DATASET

Shape: (1, 1)

Columns:
['text']

First 5 rows:
                                                text
0  First Citizen:\nBefore we proceed any further,...

VALIDATION DATASET

Shape: (1, 1)

Columns:
['text']

TEST DATASET

Shape: (1, 1)

Columns:
['text']

SEARCHING FOR TEXT COLUMN

✅ Text column found:
text

TEXT INFORMATION

Total characters: 1003854

First 1000 characters:
------------------------------------------------------------
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rath